# Covariate PCs: fit on the unrelated, project everyone else (nobody dropped)

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
for v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):   # bigsnpr parallelises itself; one BLAS thread (not OMP: it caps bigsnpr too)
    os.environ[v] = "1"
os.environ.pop("OMP_NUM_THREADS", None)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

## Functions

In [ ]:
def hexbin_safe(ax, x, y, **kw):
    """Participant density; cells with fewer than 21 people are not drawn."""
    return ax.hexbin(x, y, **{"gridsize": 120, "bins": "log", "cmap": "Greys", **kw, "mincnt": 21})

In [ ]:
STAGE = "covariate_pcs"
require_ready(cfg, STAGE)
rep, OUT = Report(cfg, STAGE), single_target(cfg, STAGE)
KEEP = single_target(cfg, "ancestry_round2")
RUN = needs_run(OUT, FORCE)
setenv(RUN=int(RUN), BED=single_target(cfg, "hm3_panel"), KEEP=KEEP, RELATEDNESS=cfg.input("relatedness"),
       KIN_CUT=cfg.param("cov_kin_cut"), K=cfg.param("cov_n_pcs"), THR_R2=cfg.param("cov_r2"),
       LD_REGIONS=cfg.input("ld_regions"), OUT=OUT, THREADS=THREADS)

## PCA (bigsnpr)

In [ ]:
%%script Rscript -
suppressMessages({ library(bigsnpr); library(data.table) })
if (Sys.getenv("RUN") != "1") { cat("exists, keeping it (FORCE = True to remake)\n"); quit(save = "no") }
env <- function(k) { v <- Sys.getenv(k); if (!nzchar(v)) stop("missing ", k); v }
ncores <- as.integer(env("THREADS"))
ncores <- max(1L, min(ncores, parallel::detectCores() - 1L))   # parallel SVD starts ncores + 1 processes
bigparallelr::set_blas_ncores(1)   # bigsnpr does the parallelism
cat("cores", ncores, "| R detects", parallel::detectCores(), "| BLAS threads", bigparallelr::get_blas_ncores(), "\n")

ld_mask <- function(map, path) {   # listed long-range LD regions
  ld <- fread(path, header = FALSE)[, V1 := sub("^chr", "", V1)]
  m <- rep(FALSE, nrow(map))
  for (r in seq_len(nrow(ld)))
    m <- m | (as.character(map$chromosome) == ld$V1[r] & map$physical.pos >= ld$V2[r] & map$physical.pos <= ld$V3[r])
  m
}

dir.create(dirname(env("OUT")), recursive = TRUE, showWarnings = FALSE)
K <- as.integer(env("K"))
obj <- bed(env("BED"))
ids <- as.character(obj$fam$sample.ID)
final <- fread(env("KEEP"), header = FALSE, colClasses = "character")$V1
rows <- which(ids %in% final)

# fit on the unrelated: both members of every related pair stay out of the fit
rel <- fread(env("RELATEDNESS"), colClasses = list(character = c("i.s", "j.s")))
rel <- rel[kin > as.numeric(env("KIN_CUT")) & i.s %in% final & j.s %in% final]
fit <- rows[!(ids[rows] %in% c(rel$i.s, rel$j.s))]
rest <- setdiff(rows, fit)
cat(length(fit), "in the fit,", length(rest), "projected\n")

svd <- bed_autoSVD(obj, ind.row = fit, ind.col = which(!ld_mask(obj$map, env("LD_REGIONS"))), k = K,
                   thr.r2 = as.numeric(env("THR_R2")), ncores = ncores)
pcs <- paste0("PC", seq_len(K))
pc_fit <- predict(svd)
pc_rest <- if (length(rest)) {
  bed_projectSelfPCA(svd, obj, ind.row = rest, ncores = ncores)$OADP_proj
} else matrix(numeric(), 0, K)
colnames(pc_fit) <- colnames(pc_rest) <- pcs

lrldr <- attr(svd, "lrldr")
jsonlite::write_json(list(n_final = length(rows), n_fit = length(fit), n_projected = length(rest),
                          n_variants_used = length(attr(svd, "subset")),
                          n_lrldr_regions = if (is.null(lrldr)) 0L else nrow(lrldr), singular_values = svd$d),
                     sub("\\.tsv$", "_meta.json", env("OUT")), auto_unbox = TRUE, digits = NA)
fwrite(rbind(data.table(person_id = ids[fit], in_fit = TRUE, pc_fit),
             data.table(person_id = ids[rest], in_fit = FALSE, pc_rest)), env("OUT"), sep = "\t")

In [ ]:
if RUN:
    mark_done(STAGE, OUT, tools={"bigsnpr": "bed_autoSVD + bed_projectSelfPCA"})

## Checks

In [ ]:
pcs = pd.read_csv(OUT, sep="\t", dtype={"person_id": str})
meta = json.load(open(OUT.replace(".tsv", "_meta.json")))
final = pd.read_csv(KEEP, header=None, dtype=str)[0]
rel = pd.read_csv(cfg.input("relatedness"), sep="\t", dtype={"i.s": str, "j.s": str})
rel = rel[(rel["kin"] > cfg.param("cov_kin_cut")) & rel["i.s"].isin(final) & rel["j.s"].isin(final)]
fit_ids = set(pcs.loc[pcs["in_fit"], "person_id"])

rep.count("n_final", len(final)); rep.count("n_fit", meta["n_fit"]); rep.count("n_projected", meta["n_projected"])
rep.note(n_variants_used=meta["n_variants_used"], n_long_range_ld_regions=meta["n_lrldr_regions"])
rep.check("every participant in the final set has PCs", set(pcs["person_id"]) == set(final))
rep.check("no related pair inside the fit set", not (rel["i.s"].isin(fit_ids) & rel["j.s"].isin(fit_ids)).any())

In [ ]:
sv = np.array(meta["singular_values"])
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].bar(range(1, len(sv) + 1), sv ** 2 / (sv ** 2).sum()); axes[0].set(xlabel="PC", ylabel="share of top-K variance")
for ax, fit in zip(axes[1:], [True, False]):
    d = pcs[pcs["in_fit"] == fit]
    hexbin_safe(ax, d["PC1"], d["PC2"], gridsize=60)
    ax.set(title="fit" if fit else "projected (has a relative)", xlabel="PC1", ylabel="PC2")
rep.fig(fig, "covariate_pcs"); plt.show()
rep.save()